# Atividade: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Bruno Andrade Zanateli (RM563736)
- Christian Souza Freitas (RM569098)
- Rodrigo Tiezzi (RM562975)

**Tema escolhido:** (escreva aqui o tema da lista do README)

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus):

1. Assistente com personalidade (Hugging Face)
2. Comparação Hugging Face x Gemini
3. Chat com memória
4. Interface web com Gradio
5. (Bônus) API com FastAPI

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [1]:
# huggingface_hub -> cliente para chamar modelos remotamente (API do Hugging Face)
# google-genai    -> SDK oficial do Google Gemini AQ.
# gradio          -> cria interfaces web a partir de funções Python
!pip install huggingface_hub google-genai gradio -q

In [2]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

# Os tokens ficam nos Secrets do Colab (ícone de chave na barra lateral)
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [9]:
# Modelos usados na atividade (os mesmos da Aula 05)
MODELO_HF = "meta-llama/Llama-3.1-8B-Instruct"
MODELO_GEMINI = "gemini-3.5-flash-lite"

cliente_hf = InferenceClient(model=MODELO_HF, token=HF_TOKEN, provider="auto")
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [4]:
# >>> Tema: Consultor de agricultura inteligente. Fala Sobre:Irrigação, umidade do solo e estações meteorológicas conectadas
# Este é o "system": a instrução que define o comportamento do assistente.
# O exemplo abaixo é do tema 1 (casa inteligente). Troque pelo tema do grupo.

SYSTEM_PROMPT = """Você é um assistente de agricultura inteligente.
Ajude o usuário com dúvidas sobre irrigação, umidade do solo e estações meteorológicas conectadas.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com agricultura inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com personalidade (Hugging Face)

Cada mensagem enviada ao modelo tem uma etiqueta `role` que diz quem escreveu:

- `system`: a regra que o assistente deve seguir
- `user`: a pergunta do usuário

Nesta etapa, a **mesma pergunta** é enviada três vezes, e **só o `system` muda**:

| Chamada | system | user |
|---|---|---|
| 1 | Especialista no tema do grupo | mesma pergunta |
| 2 | Professor para crianças | mesma pergunta |
| 3 | Resposta em uma frase | mesma pergunta |

Se as respostas saírem diferentes, a diferença veio só do `system`. É assim que se criam assistentes diferentes em cima do mesmo modelo.

In [5]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao modelo do Hugging Face e devolve o texto da resposta."""
    mensagens = [
        {"role": "system", "content": system},   # como o assistente deve se comportar
        {"role": "user",   "content": pergunta}, # o que o usuário perguntou
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [6]:
# >>> 3 personalidades: só o system muda
SYSTEM_PROMPT = """Você é um assistente de agricultura inteligente.
Ajude o usuário com dúvidas sobre irrigação, umidade do solo e estações meteorológicas conectadas.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com agricultura inteligente, diga educadamente que não pode ajudar."""


personalidades = {
    # Chamada 1: Especialista no tema do grupo
    "Especialista": SYSTEM_PROMPT,

    # Chamada 2: Professor para crianças
    "Professor para crianças": (
        "Você é um professor que explica agricultura inteligente para crianças "
        "de 10 anos. Use comparações do dia a dia e palavras simples para falar "
        "de irrigação, umidade do solo e estações meteorológicas. "
        "Responda em português, em no máximo 4 frases."
    ),

    # Chamada 3: Resposta em uma frase
    "Resposta em uma frase": (
        "Você responde qualquer pergunta sobre agricultura inteligente em uma "
        "única frase curta e direta, em português."
    ),
}


# >>> Pergunta sobre o tema do grupo
pergunta = "Como um sensor de presença pode ajudar a economizar energia em casa?"

# Temperatura baixa reduz a aleatoriedade: assim a diferença vem (quase) só do system
for nome, system in personalidades.items():
    print(f"===== {nome} =====")
    print(perguntar_hf(pergunta, system, temperatura=0.3))
    print()

===== Especialista =====
Peço desculpas, mas não sou especializado em sistemas de iluminação e não posso ajudar com essa pergunta. Se você tiver dúvidas sobre irrigação, umidade do solo ou estações meteorológicas conectadas, estou aqui para ajudar!

===== Professor para crianças =====
Um sensor de presença é como um "olho eletrônico" que detecta quando alguém está em uma determinada área. Ele pode ajudar a economizar energia em casa, por exemplo, apagando as luzes de uma sala quando ninguém está lá. É como quando você fecha a porta da cozinha para que o ar condicionado não esquente todo o apartamento.

===== Resposta em uma frase =====
Um sensor de presença pode ajudar a economizar energia em casa ao detectar a presença de pessoas em uma sala e desligar a iluminação quando não há ninguém presente, reduzindo o consumo de energia.



**Observações do grupo (Etapa 1):**

- O que mudou nas respostas de cada personalidade?
- Qual `system` gerou a resposta mais útil para o tema? Por quê?



1.   R: Mudou que cada personalidade foi condizente com suas caracteristicas. O "Especialista" nao soube responder pois sua area de conhecimento abrange somente agricultura, o "Professor para crianças" deu uma resposta com metafora para ajudar no entendimento das crianças, já o "Resposta em uma frase" foi mais objetivo em sua resposta.
2.   R: O modelo "Resposta em uma frase" teve a resposta mais util por ser mais objetiva.

---
## Etapa 2: Hugging Face x Gemini

Agora as mesmas perguntas vão para **dois modelos diferentes**.

No Gemini, o `system` não vai dentro da lista de mensagens. Ele é passado no parâmetro `system_instruction`.

In [7]:
def perguntar_gemini(pergunta, system):
    """Envia uma pergunta ao Gemini e devolve o texto da resposta."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,  # equivalente ao role "system"
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [10]:
# >>> PERSONALIZE: 3 perguntas sobre o tema do grupo.
perguntas = [
    "Como os sensores de umidade do solo podem otimizar o uso da água na irrigação?",
    "De que forma as estações meteorológicas conectadas auxiliam na previsão de necessidades de irrigação?",
    "Quais são os principais tipos de sistemas de irrigação inteligentes e como eles funcionam?",
]

for p in perguntas:
    print("PERGUNTA:", p)
    print("\n--- Hugging Face (Llama) ---")
    print(perguntar_hf(p, SYSTEM_PROMPT))
    print("\n--- Gemini ---")
    print(perguntar_gemini(p, SYSTEM_PROMPT))
    print("\n" + "=" * 60 + "\n")

PERGUNTA: Como os sensores de umidade do solo podem otimizar o uso da água na irrigação?

--- Hugging Face (Llama) ---
Os sensores de umidade do solo detectam o nível de água no solo, permitindo que a irrigação seja adaptada às necessidades reais da planta. Com essa informação, é possível evitar o desperdício de água e minimizar o impacto ambiental. Além disso, os sensores também podem detectar a presença de substâncias tóxicas no solo e alertar o agricultor para tomar ações corretivas. Isso garante uma irrigação mais eficiente e sustentável.

--- Gemini ---
Os sensores de umidade do solo medem a quantidade exata de água disponível na terra em tempo real. Com esses dados, o sistema de irrigação é acionado apenas quando necessário, evitando o desperdício por excesso de água. Isso garante que as plantas recebam a hidratação ideal para o seu desenvolvimento sem encharcar o solo. Além disso, integrados a estações meteorológicas, os sensores podem pausar a rega antes de chuvas previstas. De

**Observações do grupo (Etapa 2):**

- Os dois modelos seguiram as regras do `SYSTEM_PROMPT` (idioma, tamanho, tema)?
- Qual respondeu melhor? Em qual pergunta a diferença foi maior?



1.   R: Sim, os dois modelos seguiram as regras.
2.   R: O Gemini respondeu melhor para quem busca mais conhecimento, o hugging face foi um pouco mais superficial. A maior diferenca foi na 3° pergunta onde o Gemini nos forneceu uma explicação muito melhor.



---
## Etapa 3: Chat com memória

O modelo **não guarda memória** entre uma chamada e outra.
Quem guarda a conversa é o nosso código, na lista `historico`, que é enviada inteira a cada mensagem.

Comandos do chat:
- `sair` encerra o chat
- `limpar` apaga o histórico (o assistente "esquece" a conversa)
- `historico` mostra quantas mensagens estão guardadas

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", digite `limpar` e pergunte de novo.

In [15]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

print("Chat iniciado! Comandos: sair | limpar | historico\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]  # mantém só o system
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
    texto = resposta.choices[0].message.content

    historico.append({"role": "assistant", "content": texto})

    print(f"\nAssistente: {texto}\n")

Chat iniciado! Comandos: sair | limpar | historico

Você: limpar

(histórico apagado)

Você: sair
Encerrando chat.


In [16]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um assistente de agricultura inteligente. Ajude o usuário com dúvidas sob


**Observações do grupo (Etapa 3):**

- O que aconteceu quando vocês perguntaram o nome antes e depois do `limpar`?
- Explique, com suas palavras, por que isso acontece.

R: Antes, o segundo bloco puxou o historico e mostrou o dialogo, já o segundo não, pois o historico foi reininciado.

---
## Etapa 4: Interface web com Gradio

O assistente ganha uma interface web, com a opção de escolher o modelo (Hugging Face ou Gemini).

O Gradio entrega o histórico da conversa já no formato de `role`/`content`.
A função `texto_da_mensagem` existe porque, dependendo da versão do Gradio, o conteúdo chega como texto simples ou como lista.

In [17]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor):
    if provedor == "Hugging Face":
        # Formato HF: roles system, user e assistant
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(messages=mensagens, max_tokens=300)
        return resposta.choices[0].message.content

    else:
        # Formato Gemini: roles user e model; o system vai em system_instruction
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, max_output_tokens=300),
        )
        return resposta.text

In [19]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo")],
    title="Assistente de Agricultura",
    description="Pergunte sobre agricultura, sensores e dispositivos conectados.",
    examples=[
        ["Como funciona um sistema com sensor de umidade e irrigadores?", "Hugging Face"],
        ["Vale a pena usar automatizar o sistema de irrigação?", "Gemini"],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7831ae235d142e386f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando e coloque no README do repositório do grupo.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?

R: Eles não conseguem acessar memorias um do outro pois são IA's diferentes

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5 (Bônus): API com FastAPI

Aqui o assistente vira uma **API REST**, como no final da Aula 05.
Qualquer frontend (site, app, dispositivo IoT) poderia chamar esse endpoint.

A célula abaixo cria o arquivo `app.py`.

In [20]:
%%writefile app.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient

# O token e o system prompt vêm de variáveis de ambiente (nunca escreva o token no código)
client = InferenceClient(
    model="meta-llama/Llama-3.1-8B-Instruct",
    token=os.environ["HF_TOKEN"],
    provider="auto",
)
SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

class Pergunta(BaseModel):
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    resposta = client.chat_completion(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": pergunta.mensagem},
        ],
        max_tokens=300,
    )
    return {"resposta": resposta.choices[0].message.content}

Writing app.py


In [21]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app:app", "--port", "8000"])
time.sleep(5)  # espera o servidor subir
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [22]:
# Testa o endpoint como um frontend faria (HTTP POST com JSON)
import requests

r = requests.post("http://localhost:8000/chat", json={"mensagem": "O que é IoT?"})
print(r.status_code)
print(r.json()["resposta"])

200
Peço desculpas, mas não posso ajudar com essa pergunta. O meu foco é ajudar com dúvidas sobre agricultura inteligente, como irrigação, umidade do solo e estações meteorológicas conectadas. Se precisar de ajuda com IoT em geral, posso sugerir uma outra fonte de informação.


In [23]:
# Encerra o servidor
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.
